In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
ROOT_DIR = Path("..")

PROCESSED_DIR = ROOT_DIR / "data" / "processed"

KPI_DIR = ROOT_DIR / "data" / "processed" / "dashboard"

KPI_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
rossmann = pd.read_csv(
    PROCESSED_DIR / "rossmann_clean.csv",
    low_memory=False
)

superstore = pd.read_csv(
    PROCESSED_DIR / "superstore_clean.csv",
    encoding="latin1",
    low_memory=False
)

In [4]:
executive_kpis = pd.DataFrame({
    "Metric": [
        "Total Revenue",
        "Total Customers",
        "Total Orders",
        "Total Profit"
    ],
    "Value": [
        superstore["Sales"].sum(),
        superstore["Customer ID"].nunique(),
        superstore["Order ID"].nunique(),
        superstore["Profit"].sum()
    ]
})

executive_kpis

,Metric,Value
0,Total Revenue,2.297201e+06
1,Total Customers,7.930000e+02
2,Total Orders,5.009000e+03
3,Total Profit,2.863970e+05


In [5]:
executive_kpis.to_csv(
    KPI_DIR / "executive_kpis.csv",
    index=False
)

In [6]:
revenue_kpis = (
    superstore
    .groupby("Order Date")
    ["Sales"]
    .sum()
    .reset_index()
)

revenue_kpis.head()

,Order Date,Sales
0,2014-01-03,16.448
1,2014-01-04,288.060
2,2014-01-05,19.536
3,2014-01-06,4407.100
4,2014-01-07,87.158


In [7]:
revenue_kpis.to_csv(
    KPI_DIR / "revenue_kpis.csv",
    index=False
)

In [8]:
customer_segments = pd.read_csv(
    ROOT_DIR /
    "models" /
    "kmeans" /
    "customer_segments.csv"
)

In [9]:
customer_kpis = (
    customer_segments
    .groupby("segment")
    .agg({
        "total_sales":"mean",
        "total_profit":"mean",
        "total_orders":"mean"
    })
    .reset_index()
)

customer_kpis

,segment,total_sales,total_profit,total_orders
0,0,2874.505417,328.128142,7.018127
1,1,9951.135073,2502.723953,7.800000
2,2,4583.976467,421.793146,9.912281
3,3,1238.828950,56.368350,3.976898


In [10]:
customer_kpis.to_csv(
    KPI_DIR / "customer_kpis.csv",
    index=False
)

In [11]:
forecast = pd.read_csv(
    ROOT_DIR /
    "models" /
    "prophet" /
    "forecast.csv"
)

In [12]:
forecast_kpis = forecast[
[
    "ds",
    "yhat"
]]

In [13]:
forecast_kpis.to_csv(
    KPI_DIR /
    "forecast_kpis.csv",
    index=False
)

In [14]:
inventory = pd.read_csv(
    ROOT_DIR /
    "data" /
    "external" /
    "inventory_data.csv"
)

In [15]:
inventory_kpis = pd.DataFrame({
    "Total Products":[
        inventory["product_id"].nunique()
    ],
    "Total Stock":[
        inventory["current_stock"].sum()
    ],
    "Average Stock":[
        inventory["current_stock"].mean()
    ]
})

inventory_kpis

,Total Products,Total Stock,Average Stock
0,1862,990585,523.012144


In [16]:
inventory_kpis.to_csv(
    KPI_DIR /
    "inventory_kpis.csv",
    index=False
)

In [17]:
top_products = (
    superstore
    .groupby("Product Name")
    ["Sales"]
    .sum()
    .sort_values(
        ascending=False
    )
    .head(20)
    .reset_index()
)

top_products

,Product Name,Sales
0,Canon imageCLASS 2200 Advanced Copier,61599.8240
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.3840
2,Cisco TelePresence System EX90 Videoconferenci...,22638.4800
3,HON 5400 Series Task Chairs for Big and Tall,21870.5760
4,GBC DocuBind TL300 Electric Binding System,19823.4790
5,GBC Ibimaster 500 Manual ProClick Binding System,19024.5000
6,Hewlett Packard LaserJet 3310 Copier,18839.6860
7,HP Designjet T520 Inkjet Large Format Printer ...,18374.8950
8,GBC DocuBind P400 Electric Binding System,17965.0680
9,High Speed Automatic Electric Letter Opener,17030.3120


In [18]:
top_products.to_csv(
    KPI_DIR /
    "top_products.csv",
    index=False
)

In [19]:
top_customers = (
    superstore
    .groupby("Customer Name")
    ["Sales"]
    .sum()
    .sort_values(
        ascending=False
    )
    .head(20)
    .reset_index()
)

top_customers

,Customer Name,Sales
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571
5,Ken Lonsdale,14175.229
6,Sanjit Chand,14142.334
7,Hunter Lopez,12873.298
8,Sanjit Engle,12209.438
9,Christopher Conant,12129.072


In [20]:
top_customers.to_csv(
    KPI_DIR /
    "top_customers.csv",
    index=False
)

In [21]:
import os

files = os.listdir(KPI_DIR)

for file in files:
    print(file)

customer_kpis.csv
executive_kpis.csv
forecast_kpis.csv
inventory_kpis.csv
revenue_kpis.csv
top_customers.csv
top_products.csv


In [22]:
segment_counts = pd.read_csv(
    ROOT_DIR /
    "models" /
    "kmeans" /
    "customer_segments.csv"
)

segment_counts = (
    segment_counts["segment"]
    .value_counts()
    .reset_index()
)

segment_counts.columns = [
    "segment",
    "count"
]

segment_counts.to_csv(
    KPI_DIR /
    "segment_counts.csv",
    index=False
)

In [34]:
segment_counts.head(5)

,segment,count
0,0,331
1,3,303
2,2,114
3,1,45


In [23]:
importance = pd.read_csv(
    ROOT_DIR /
    "models" /
    "xgboost" /
    "feature_importance.csv"
)

importance.to_csv(
    KPI_DIR /
    "revenue_drivers.csv",
    index=False
)

In [35]:
importance.head()

,Feature,Importance
0,Customers,0.816619
1,SalesPerCustomer,0.117681
2,Promo,0.017624
3,PromoActive,0.016346
4,IsHoliday,0.006929


In [24]:
forecast = pd.read_csv(
    ROOT_DIR /
    "models" /
    "prophet" /
    "forecast.csv"
)

forecast.tail(90).to_csv(
    KPI_DIR /
    "forecast_90_days.csv",
    index=False
)

In [25]:
anomalies = pd.read_csv(
    ROOT_DIR /
    "models" /
    "isolation_forest" /
    "anomalies.csv"
)

anomaly_kpi = pd.DataFrame({
    "Total_Anomalies":[len(anomalies)]
})

anomaly_kpi.to_csv(
    KPI_DIR /
    "anomaly_kpi.csv",
    index=False
)

In [26]:
forecast = pd.read_csv(
    ROOT_DIR /
    "models" /
    "prophet" /
    "forecast.csv"
)

forecast.to_csv(
    KPI_DIR /
    "full_forecast.csv",
    index=False
)

In [28]:
monthly_revenue = (
    revenue_kpis
    .copy()
)

monthly_revenue["Order Date"] = pd.to_datetime(
    monthly_revenue["Order Date"]
)

monthly_revenue = (
    monthly_revenue
    .groupby(
        pd.Grouper(
            key="Order Date",
            freq="ME"
        )
    )["Sales"]
    .sum()
    .reset_index()
)

monthly_revenue.to_csv(
    KPI_DIR /
    "monthly_revenue.csv",
    index=False
)

In [29]:
segments = pd.read_csv(
    ROOT_DIR /
    "models" /
    "kmeans" /
    "customer_segments.csv"
)

segments.to_csv(
    KPI_DIR /
    "customer_segments_full.csv",
    index=False
)

In [30]:
inventory = pd.read_csv(
    ROOT_DIR /
    "data" /
    "external" /
    "inventory_data.csv"
)

inventory.to_csv(
    KPI_DIR /
    "inventory_full.csv",
    index=False
)

In [31]:
competitors = pd.read_csv(
    ROOT_DIR /
    "data" /
    "external" /
    "competitor_data.csv"
)

competitors.to_csv(
    KPI_DIR /
    "competitor_full.csv",
    index=False
)

In [32]:
segments

,customer_id,total_sales,total_profit,total_quantity,total_orders,segment
0,AA-10315,5563.560,-362.8825,30,5,0
1,AA-10375,1056.390,277.3824,41,9,0
2,AA-10480,1790.512,435.8274,36,4,3
3,AA-10645,5086.935,857.8033,64,6,0
4,AB-10015,886.156,129.3465,13,3,3
...,...,...,...,...,...,...
788,XP-21865,2374.658,621.2300,100,11,2
789,YC-21895,5454.350,1305.6290,31,5,0
790,YS-21880,6720.444,1778.2923,58,8,1
791,ZC-21910,8025.707,-1032.1490,105,13,2
